# Conformal Triage — Phase 4: core ablation tier

Runs the cheap ablations promised in Section 4 of the paper: classification metrics and
per-class sensitivity, reference rules (softmax threshold and temperature scaling),
APS vs RAPS, the domain ablation (ISIC-only head) and the AK-benign convention with its audit.

Requires what Phase 3 left in your Drive (`emb/` and `results/head_weights.npz`).
**CPU is enough.** ~15–20 min. The outputs feed Section 5 (ablations) of the paper.


In [ ]:
# 1) Setup: Drive, embeddings, metadata, Phase-3 head, probabilities and APS scores
SEED = 2026
import numpy as np, pandas as pd, os, json
from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/conformal-triage'

d = np.load(f'{BASE}/emb/hiba_emb.npz'); Eh = dict(zip(d['ids'], d['features'].astype(np.float32)))
d = np.load(f'{BASE}/emb/pad_emb.npz');  Ep = dict(zip(d['ids'], d['features'].astype(np.float32)))
Xi, yi = [], []
for k in range(3):
    d = np.load(f'{BASE}/emb/isic2019_emb_part{k}.npz')
    Xi.append(d['features'].astype(np.float32)); yi.append(d['labels'])
Xisic = np.concatenate(Xi); yisic = np.concatenate(yi).astype(int)
Hm = pd.read_csv(f'{BASE}/emb/hiba_isic_metadata.csv')
Pm = pd.read_csv(f'{BASE}/emb/pad_isic_metadata.csv')
w = np.load(f'{BASE}/results/head_weights.npz')
COEF, INTC, SC_M, SC_S = w['coef'], w['intercept'], w['scaler_mean'], w['scaler_scale']
def probs_of(X):
    z = (X - SC_M) / SC_S @ COEF.T + INTC
    z -= z.max(1, keepdims=True); e = np.exp(z); return e / e.sum(1, keepdims=True)
print('loaded: Phase-3 head + embeddings + metadata')


In [ ]:
# 2) Pipeline functions (Phase-3 core)
import numpy as np, pandas as pd

CLS = ['MEL','NV','BCC','AK','BKL','DF','VASC','SCC']
MAL = {'MEL','BCC','SCC','AK'}
ROMAN = {'I':1,'II':2,'III':3,'IV':4,'V':5,'VI':6}
HIBA_MAP = {'melanoma':'MEL','nevus':'NV','basal cell carcinoma':'BCC',
            'squamous cell carcinoma':'SCC','actinic keratosis':'AK',
            'seborrheic keratosis':'BKL','solar lentigo':'BKL',
            'lichenoid keratosis':'BKL','dermatofibroma':'DF','vascular lesion':'VASC'}
DX3_MAP = {'Melanoma, NOS':'MEL','Melanoma':'MEL','Nevus':'NV',
           'Basal cell carcinoma':'BCC','Squamous cell carcinoma, NOS':'SCC',
           'Squamous cell carcinoma':'SCC','Solar or actinic keratosis':'AK',
           'Seborrheic keratosis':'BKL','Solar lentigo':'BKL',
           'Lichen planus like keratosis':'BKL','Lichenoid keratosis':'BKL',
           'Dermatofibroma':'DF','Hemangioma':'VASC','Angioma':'VASC',
           'Vascular lesion':'VASC',
           'Benign soft tissue proliferations - Vascular':'VASC'}

def lesion_table(meta, cohort):
    """One row per lesion: cls, patient, g (phototype 1..6 or NaN), list of isic_id."""
    df = meta.copy()
    if 'diagnosis' in df.columns and df['diagnosis'].notna().any():
        dx = df['diagnosis']            # old archive schema (legacy export)
    else:                               # new schema: diagnosis_1/2/3; vascular lesions
        d2 = df['diagnosis_2'] if 'diagnosis_2' in df.columns else pd.Series(np.nan, index=df.index)
        dx = df['diagnosis_3'].fillna(d2)  # vascular lesions live in diagnosis_2
    df['cls'] = dx.map({**HIBA_MAP, **DX3_MAP})
    df['g'] = df['fitzpatrick_skin_type'].map(ROMAN)
    missing = sorted(dx[df['cls'].isna()].dropna().unique().tolist()) + (['(empty)'] if (df['cls'].isna() & dx.isna()).any() else [])
    assert not missing, f'{cohort}: unmapped diagnoses: {missing} (add them to the map)'
    les = df.groupby('lesion_id').agg(cls=('cls','first'), patient=('patient_id','first'),
                                      g=('g','first'), imgs=('isic_id', lambda s: tuple(s))).reset_index()
    chk = df.groupby('lesion_id').agg(nc=('cls','nunique'), npat=('patient_id','nunique'))
    assert (chk.nc == 1).all() and (chk.npat == 1).all(), f'{cohort}: lesion with inconsistent class or patient'
    les['mal'] = les.cls.isin(MAL)
    return les

def membership(les):
    """patient -> set of strata: ('M', g) for each phototyped malignant lesion; ('B',) for each phototyped benign one."""
    m = {}
    ph = les[les.g.notna()]
    for pid, sub in ph.groupby('patient'):
        s = set()
        for _, r in sub.iterrows():
            s.add(('M', int(r.g)) if r.mal else ('B',))
        m[pid] = s
    return m

def floor_n(alpha):
    return int(np.ceil(1.0/alpha - 1e-9)) - 1

def pad_training_draw(les, rng, train_frac=0.4, cal_frac=0.6, alpha=0.05, max_tries=100):
    """40% of patients to training, stratified by phototype (incl. the no-phototype group),
    verified: every enforced stratum keeps enough patients in the remainder for the
    calibration fraction to clear the floor in expectation. Re-drawn if it fails."""
    pf = les.groupby('patient')['g'].agg(lambda s: tuple(s.dropna().unique()))
    pgroup = {p: (int(v[0]) if len(v) else 0) for p, v in pf.items()}
    pats = np.array(sorted(pgroup)); grp = np.array([pgroup[p] for p in pats])
    mem = membership(les)
    enforced = [('M',1), ('M',2), ('M',3), ('B',)]
    fl = floor_n(alpha)
    for _ in range(max_tries):
        tr = set()
        for g in np.unique(grp):
            idx = np.where(grp == g)[0]
            tr.update(pats[rng.choice(idx, int(round(train_frac*len(idx))), replace=False)])
        rem = [p for p in pats if p not in tr]
        ok = all(cal_frac*sum(1 for p in rem if s in mem.get(p, ())) >= fl for s in enforced)
        if ok:
            return sorted(tr), sorted(rem)
    raise RuntimeError('could not verify the training draw')

def aps_scores(P):
    """S[i, c] = sum of probabilities >= p_c (inclusive), non-randomised APS."""
    order = np.argsort(-P, axis=1)
    cum = np.take_along_axis(P, order, 1).cumsum(1)
    S = np.empty_like(P)
    np.put_along_axis(S, order, cum, 1)
    return S

def qhat(scores, alpha):
    n = len(scores)
    k = int(np.ceil((n + 1) * (1 - alpha) - 1e-9))
    if n == 0 or k > n:
        return np.inf
    return np.sort(scores)[k - 1]

def decluster(les_cal, mem_cal, rng):
    """Per stratum, one lesion per patient (uniform among that patient's lesions in the stratum).
    Returns dict stratum -> lesion indices (positions in les_cal)."""
    out = {}
    ph = les_cal[les_cal.g.notna()]
    for s in sorted(set().union(*mem_cal.values()) if mem_cal else set(), key=str):  # fixed order
        rows = ph[(ph.mal & (ph.g == s[1]))] if s[0] == 'M' else ph[~ph.mal]
        picks = [rng.choice(sub.index.values) for _, sub in rows.groupby('patient')]
        out[s] = np.array(sorted(picks), dtype=int)
    return out

def evaluate_draw(les, S, cal_pats, test_pats, alphas, rng):
    """S: APS score matrix aligned to the index of `les`. Returns records per (alpha, stratum)."""
    cal = les[les.patient.isin(cal_pats)]
    test = les[les.patient.isin(test_pats) & les.g.notna()]
    mem_cal = membership(cal)
    cells = decluster(cal, mem_cal, rng)
    strata = sorted(cells, key=str)
    recs = []
    for a in alphas:
        q = {}
        for s in strata:
            idx = cells[s]
            q[s] = qhat(S[idx, [CLS.index(les.loc[i, 'cls']) for i in idx]], a) if len(idx) else np.inf
        qB = q.get(('B',), np.inf)
        Ct = np.zeros((len(test), len(CLS)), bool)
        Stest = S[test.index.values]
        for c, name in enumerate(CLS):
            if name in MAL:
                thr = np.array([q.get(('M', int(g)), np.inf) for g in test.g.values])
            else:
                thr = np.full(len(test), qB)
            Ct[:, c] = Stest[:, c] <= thr
        ytrue = np.array([CLS.index(c) for c in test.cls.values])
        covered = Ct[np.arange(len(test)), ytrue]
        single_benign = (Ct.sum(1) == 1) & ~Ct[:, [CLS.index(c) for c in CLS if c in MAL]].any(1)
        observe = single_benign
        for s in strata:
            n_cal = len(cells[s])
            if s[0] == 'M':
                sel = test.mal.values & (test.g.values == s[1])
            else:
                sel = ~test.mal.values
            if sel.sum() == 0 and n_cal == 0:
                continue
            miss = float((observe[sel] & test.mal.values[sel]).mean()) if s[0] == 'M' and sel.sum() else np.nan
            if sel.sum():
                pw = pd.DataFrame({'p': test.patient.values[sel], 'cov': covered[sel],
                                   'ref': ~observe[sel]}).groupby('p').mean().mean()
            recs.append(dict(alpha=a, stratum=f'{s[0]}{s[1] if s[0]=="M" else ""}',
                             n_cal=n_cal, degenerate=bool(np.isinf(q[s])),
                             coverage=float(covered[sel].mean()) if sel.sum() else np.nan,
                             coverage_pw=float(pw['cov']) if sel.sum() else np.nan,
                             miss_rate=miss,
                             referral=float((~observe[sel]).mean()) if sel.sum() else np.nan,
                             referral_pw=float(pw['ref']) if sel.sum() else np.nan,
                             set_size=float(Ct[sel].sum(1).mean()) if sel.sum() else np.nan,
                             n_test=int(sel.sum()), n_test_pat=int(test.patient.values[sel].size and len(set(test.patient.values[sel])))))
        # marginal baseline: a single global threshold (global de-clustering: one lesion per patient)
        picks = [rng.choice(sub.index.values) for _, sub in cal[cal.g.notna()].groupby('patient')]
        picks = np.array(sorted(picks), dtype=int)
        qm = qhat(S[picks, [CLS.index(les.loc[i, 'cls']) for i in picks]], a)
        Cm = Stest <= qm
        obs_m = (Cm.sum(1) == 1) & ~Cm[:, [CLS.index(c) for c in CLS if c in MAL]].any(1)
        for g in sorted(test.g.dropna().unique()):
            sel = test.mal.values & (test.g.values == g)
            if sel.sum():
                recs.append(dict(alpha=a, stratum=f'marginal-M{int(g)}', n_cal=len(picks),
                                 degenerate=bool(np.isinf(qm)),
                                 coverage=float(Cm[np.arange(len(test)), ytrue][sel].mean()),
                                 miss_rate=float((obs_m[sel]).mean()),
                                 referral=float((~obs_m[sel]).mean()),
                                 set_size=float(Cm[sel].sum(1).mean()), n_test=int(sel.sum())))
    return recs


In [ ]:
# 2b) Phase-4 variants: parametrised mal_set, RAPS, threshold rule, temperature, per-class metrics
from scipy.optimize import minimize_scalar
from sklearn.metrics import balanced_accuracy_score, f1_score, roc_auc_score

def build_mal(les, ak_malignant=True):
    M = frozenset({'MEL','BCC','SCC','AK'} if ak_malignant else {'MEL','BCC','SCC'})
    l = les.copy(); l['mal'] = l.cls.isin(M); return l, M

def raps_scores(P, lam, kreg):
    order = np.argsort(-P, axis=1); Ps = np.take_along_axis(P, order, 1); cum = Ps.cumsum(1)
    reg = lam * np.maximum(0, np.arange(1, P.shape[1] + 1)[None, :] - kreg)
    S = np.empty_like(P); np.put_along_axis(S, order, cum + reg, 1); return S

def tune_raps(Ptr, ytr, alpha=0.05, seed=SEED, tol=0.005):
    # tol: coverage on the tuning half is a noisy estimate (~2.5 standard errors at
    # this n); the actual guarantee never depends on this choice, which is fixed
    # before seeing the calibration set. If no config passes, the highest-coverage one is taken.
    r = np.random.default_rng(seed); n = len(ytr); idx = r.permutation(n)
    a, b = idx[:n // 2], idx[n // 2:]
    res = []
    for lam in [0.001, 0.01, 0.1, 0.5]:
        for k in [1, 2, 3, 5]:
            S = raps_scores(Ptr, lam, k); q = qhat(S[a, ytr[a]], alpha); sets = S[b] <= q
            cov = sets[np.arange(len(b)), ytr[b]].mean(); sz = sets.sum(1).mean()
            res.append(((lam, k), float(cov), float(sz)))
    ok = [x for x in res if x[1] >= 1 - alpha - tol]
    el = min(ok, key=lambda x: x[2]) if ok else max(res, key=lambda x: x[1])
    print(f'RAPS tuning: params={el[0]} cov_tune={el[1]:.3f} size_tune={el[2]:.2f}')
    return el[0]

def evaluate_draw2(les, S, cal_pats, test_pats, alphas, rng, mal_set, per_class=False):
    """Phase-3 evaluate_draw with a parametrised malignant partition and optional per-class
    rows: coverage per true class and referral sensitivity per malignant class."""
    MALc = [CLS.index(c) for c in CLS if c in mal_set]
    cal = les[les.patient.isin(cal_pats)]
    test = les[les.patient.isin(test_pats) & les.g.notna()]
    mem_cal = {}
    for pid, sub in cal[cal.g.notna()].groupby('patient'):
        s = set()
        for _, rr in sub.iterrows(): s.add(('M', int(rr.g)) if rr.mal else ('B',))
        mem_cal[pid] = s
    cells = {}
    ph = cal[cal.g.notna()]
    for s in sorted(set().union(*mem_cal.values()) if mem_cal else set(), key=str):  # fixed order
        rows = ph[(ph.mal & (ph.g == s[1]))] if s[0] == 'M' else ph[~ph.mal]
        cells[s] = np.array(sorted(rng.choice(sub.index.values) for _, sub in rows.groupby('patient')), dtype=int)
    strata = sorted(cells, key=str)
    recs = []
    ytrue = np.array([CLS.index(c) for c in test.cls.values])
    Stest = S[test.index.values]
    for a in alphas:
        q = {s: (qhat(S[cells[s], [CLS.index(les.loc[i, 'cls']) for i in cells[s]]], a) if len(cells[s]) else np.inf) for s in strata}
        qB = q.get(('B',), np.inf)
        Ct = np.zeros((len(test), len(CLS)), bool)
        for c, name in enumerate(CLS):
            thr = np.array([q.get(('M', int(g)), np.inf) for g in test.g.values]) if name in mal_set else np.full(len(test), qB)
            Ct[:, c] = Stest[:, c] <= thr
        covered = Ct[np.arange(len(test)), ytrue]
        observe = (Ct.sum(1) == 1) & ~Ct[:, MALc].any(1)
        for s in strata:
            sel = (test.mal.values & (test.g.values == s[1])) if s[0] == 'M' else ~test.mal.values
            if sel.sum() == 0 and len(cells[s]) == 0: continue
            pw = (pd.DataFrame({'p': test.patient.values[sel], 'cov': covered[sel], 'ref': ~observe[sel]})
                  .groupby('p').mean().mean()) if sel.sum() else None
            recs.append(dict(alpha=a, stratum=f"{s[0]}{s[1] if s[0]=='M' else ''}", n_cal=len(cells[s]),
                             degenerate=bool(np.isinf(q[s])),
                             coverage=float(covered[sel].mean()) if sel.sum() else np.nan,
                             coverage_pw=float(pw['cov']) if sel.sum() else np.nan,
                             miss_rate=float((observe[sel]).mean()) if s[0] == 'M' and sel.sum() else np.nan,
                             referral=float((~observe[sel]).mean()) if sel.sum() else np.nan,
                             referral_pw=float(pw['ref']) if sel.sum() else np.nan,
                             set_size=float(Ct[sel].sum(1).mean()) if sel.sum() else np.nan,
                             n_test=int(sel.sum())))
        if per_class:
            for c, name in enumerate(CLS):
                sel = (ytrue == c)
                if sel.sum():
                    recs.append(dict(alpha=a, stratum=f'cov-{name}', coverage=float(covered[sel].mean()), n_test=int(sel.sum())))
                    if name in mal_set:
                        recs.append(dict(alpha=a, stratum=f'sens-{name}', referral=float((~observe[sel]).mean()), n_test=int(sel.sum())))
    return recs

def temp_fit(P, y):
    lg = np.log(np.clip(P, 1e-12, 1))
    def nll(T):
        z = lg / T; z = z - z.max(1, keepdims=True); pr = np.exp(z); pr /= pr.sum(1, keepdims=True)
        return -np.log(np.clip(pr[np.arange(len(y)), y], 1e-12, 1)).mean()
    return float(minimize_scalar(nll, bounds=(0.25, 10), method='bounded').x)

def rule_eval(P, les, cal_pats, test_pats, alphas, mal_set, rng, T=None):
    cal = les[les.patient.isin(cal_pats) & les.g.notna()]
    picks = np.array(sorted(rng.choice(s.index.values) for _, s in cal.groupby('patient')), dtype=int)
    if T is None: T = 1.0
    elif T == 'fit': T = temp_fit(P[picks], np.array([CLS.index(c) for c in les.loc[picks, 'cls']]))
    lg = np.log(np.clip(P, 1e-12, 1)) / T; lg -= lg.max(1, keepdims=True)
    Pt = np.exp(lg); Pt /= Pt.sum(1, keepdims=True)
    top, pred = Pt.max(1), Pt.argmax(1)
    pred_ben = ~np.isin(np.array(CLS)[pred], list(mal_set))
    test = les[les.patient.isin(test_pats) & les.g.notna()]
    mal_c = les.mal.values[picks]
    out = []
    for a in alphas:
        tau = 1.01
        for t in np.concatenate([[0.0], np.sort(top[picks])]):
            o = pred_ben[picks] & (top[picks] >= t)
            if (o & mal_c).sum() / max(mal_c.sum(), 1) <= a: tau = t; break
        ti = test.index.values
        obs = pred_ben[ti] & (top[ti] >= tau)
        for g in sorted(test.g.dropna().unique()):
            sel = test.mal.values & (test.g.values == g)
            if sel.sum():
                out.append(dict(alpha=a, group=int(g), tau=float(tau), T=float(T),
                                miss=float(obs[sel].mean()), n_mal=int(sel.sum())))
        selb = ~test.mal.values
        out.append(dict(alpha=a, group=0, tau=float(tau), T=float(T),
                        miss=np.nan, release=float(obs[selb].mean()), n_ben=int(selb.sum())))
    return out


In [ ]:
# 3) Tables and splits (identical to Phase 3 by seed)
hles = lesion_table(Hm, 'hiba').reset_index(drop=True)
ples = lesion_table(Pm, 'pad').reset_index(drop=True)
assert len(hles) == 1246 and len(ples) == 1891
rng = np.random.default_rng(SEED)
tr, rem = pad_training_draw(ples, rng)
def lesion_X(les, E): return np.stack([np.mean([E[i] for i in t], 0) for t in les.imgs])
Ph, Pp = probs_of(lesion_X(hles, Eh)), probs_of(lesion_X(ples, Ep))
Sh, Sp = aps_scores(Ph), aps_scores(Pp)
hp = sorted(membership(hles))
ALPHAS = [0.05, 0.10, 0.15]
print(f'training draw reproduced ({len(tr)} patients); scores ready')


In [ ]:
# 4) Classification metrics + per-class sensitivity (primary configuration, same 100 draws)
hlesM, MSET = build_mal(hles, True); plesM, _ = build_mal(ples, True)
cls_rows, pc_rows = [], []
for d in range(100):
    r = np.random.default_rng(SEED * 1000 + d)
    cal = set(r.choice(hp, int(round(0.7 * len(hp))), replace=False))
    pc_rows += [dict(cohort='hiba', draw=d, **x) for x in
                evaluate_draw2(hlesM, Sh, cal, set(hp) - cal, ALPHAS, r, MSET, per_class=True) if x['stratum'].startswith(('cov-', 'sens-'))]
    calp = set(r.choice(rem, int(round(0.6 * len(rem))), replace=False))
    pc_rows += [dict(cohort='pad', draw=d, **x) for x in
                evaluate_draw2(plesM, Sp, calp, set(rem) - calp, ALPHAS, r, MSET, per_class=True) if x['stratum'].startswith(('cov-', 'sens-'))]
    for name, les, Pr, tpats in [('hiba', hlesM, Ph, set(hp) - cal), ('pad', plesM, Pp, set(rem) - calp)]:
        t = les[les.patient.isin(tpats) & les.g.notna()]
        y = np.array([CLS.index(c) for c in t.cls]); pr = Pr[t.index.values]; yhat = pr.argmax(1)
        aucs = []
        for c in range(8):
            pos = (y == c).sum()
            if 0 < pos < len(y):
                aucs.append(roc_auc_score((y == c).astype(int), pr[:, c]))
        auc = float(np.mean(aucs)) if aucs else np.nan
        cls_rows.append(dict(cohort=name, draw=d, top1=float((yhat == y).mean()),
                             bal_acc=float(balanced_accuracy_score(y, yhat)),
                             macro_f1=float(f1_score(y, yhat, average='macro', labels=np.arange(8), zero_division=0)),
                             auc_ovr=auc, n=len(y)))
pd.DataFrame(cls_rows).to_csv(f'{BASE}/results/class_metrics.csv', index=False)
pd.DataFrame(pc_rows).to_csv(f'{BASE}/results/perclass_cov_sens.csv', index=False)
C = pd.DataFrame(cls_rows).groupby('cohort').mean(numeric_only=True).round(3)
print(C[['top1', 'bal_acc', 'macro_f1', 'auc_ovr']].to_string())


In [ ]:
# 5) Rule baselines: softmax threshold and temperature scaling (100 draws, own seeded stream)
rb = []
for d in range(100):
    r = np.random.default_rng((SEED + 7) * 1000 + d)
    cal = set(r.choice(hp, int(round(0.7 * len(hp))), replace=False))
    calp = set(r.choice(rem, int(round(0.6 * len(rem))), replace=False))
    for name, les, Pr, cpats, tpats in [('hiba', hlesM, Ph, cal, set(hp) - cal), ('pad', plesM, Pp, calp, set(rem) - calp)]:
        r1 = np.random.default_rng((SEED + 8) * 1000 + d)
        rb += [dict(cohort=name, draw=d, variant='softmax', **x) for x in rule_eval(Pr, les, cpats, tpats, ALPHAS, MSET, r1, T=None)]
        r2 = np.random.default_rng((SEED + 8) * 1000 + d)
        rb += [dict(cohort=name, draw=d, variant='temp', **x) for x in rule_eval(Pr, les, cpats, tpats, ALPHAS, MSET, r2, T='fit')]
RB = pd.DataFrame(rb); RB.to_csv(f'{BASE}/results/rules_baselines.csv', index=False)
RB['excede'] = RB.miss > RB.alpha
res = RB[RB.group > 0].groupby(['cohort', 'variant', 'alpha', 'group']).agg(
    miss_medio=('miss', 'mean'), excede_alfa_pct=('excede', lambda s: 100 * s.mean())).round(3)
print(res.to_string())


In [ ]:
# 6) APS vs RAPS (hyperparameters selected on the head-training split)
pad_tr = ples[ples.patient.isin(tr)]
Xpt = np.stack([Ep[i] for t in pad_tr.imgs for i in t])
ypt = np.array([CLS.index(rr.cls) for _, rr in pad_tr.iterrows() for _ in rr.imgs])
Ptrain = probs_of(np.vstack([Xisic, Xpt])); ytrain = np.concatenate([yisic, ypt])
lam, kreg = tune_raps(Ptrain, ytrain, alpha=0.05)
open(f'{BASE}/results/raps_params.txt', 'w').write(f'lambda={lam} k_reg={kreg} (selected on train, alpha=0.05, seed={SEED})')
print('RAPS: lambda', lam, 'k_reg', kreg)
Shr, Spr = raps_scores(Ph, lam, kreg), raps_scores(Pp, lam, kreg)
rr_ = []
for d in range(100):
    r = np.random.default_rng(SEED * 1000 + d)
    cal = set(r.choice(hp, int(round(0.7 * len(hp))), replace=False))
    rr_ += [dict(cohort='hiba', draw=d, **x) for x in evaluate_draw2(hlesM, Shr, cal, set(hp) - cal, ALPHAS, r, MSET, per_class=True)]
    calp = set(r.choice(rem, int(round(0.6 * len(rem))), replace=False))
    rr_ += [dict(cohort='pad', draw=d, **x) for x in evaluate_draw2(plesM, Spr, calp, set(rem) - calp, ALPHAS, r, MSET, per_class=True)]
RR = pd.DataFrame(rr_); RR.to_csv(f'{BASE}/results/raps_results.csv', index=False)
m = RR[(RR.alpha == 0.05) & (~RR.stratum.str.startswith(('cov-', 'sens-')))].groupby(['cohort', 'stratum'])[['coverage', 'set_size', 'referral']].mean().round(3)
print(m.to_string())


In [ ]:
# 7) Domain ablation: head trained on ISIC 2019 only
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
sc2 = StandardScaler().fit(Xisic)
head2 = LogisticRegression(max_iter=1000, class_weight='balanced').fit(sc2.transform(Xisic), yisic)
def probs2(X):
    z = sc2.transform(X) @ head2.coef_.T + head2.intercept_
    z -= z.max(1, keepdims=True); e = np.exp(z); return e / e.sum(1, keepdims=True)
ba = balanced_accuracy_score(yisic, head2.predict(sc2.transform(Xisic)))
open(f'{BASE}/results/domain_head_report.txt', 'w').write(f'ISIC-only head: balanced acc (train) = {ba:.4f}\n')
Ph2, Pp2 = probs2(lesion_X(hles, Eh)), probs2(lesion_X(ples, Ep))
Sh2, Sp2 = aps_scores(Ph2), aps_scores(Pp2)
dr = []
for d in range(100):
    r = np.random.default_rng(SEED * 1000 + d)
    cal = set(r.choice(hp, int(round(0.7 * len(hp))), replace=False))
    dr += [dict(cohort='hiba', draw=d, **x) for x in evaluate_draw2(hlesM, Sh2, cal, set(hp) - cal, ALPHAS, r, MSET)]
    calp = set(r.choice(rem, int(round(0.6 * len(rem))), replace=False))
    dr += [dict(cohort='pad', draw=d, **x) for x in evaluate_draw2(plesM, Sp2, calp, set(rem) - calp, ALPHAS, r, MSET)]
DR = pd.DataFrame(dr); DR.to_csv(f'{BASE}/results/domain_results.csv', index=False)
m = DR[(DR.alpha == 0.05)].groupby(['cohort', 'stratum'])[['coverage', 'set_size', 'referral']].mean().round(3)
print('ISIC-only head (a=0.05):'); print(m.to_string())


In [ ]:
# 8) AK-benign convention: recomputed audit + evaluation (same head, same draws)
hlesB, MSETB = build_mal(hles, False); plesB, _ = build_mal(ples, False)
def audit_deg(l, cohort, draws=4000, seed=SEED):
    mem = {}
    for pid, sub in l[l.g.notna()].groupby('patient'):
        s = set()
        for _, rr in sub.iterrows(): s.add(('M', int(rr.g)) if rr.mal else ('B',))
        mem[pid] = s
    r = np.random.default_rng(seed)
    strata = sorted(set().union(*mem.values()), key=str)
    counts = {s: [] for s in strata}
    if cohort == 'hiba':
        pats = np.array(sorted(mem))
        for _ in range(draws):
            cal = set(r.choice(pats, int(round(0.7 * len(pats))), replace=False))
            for s in strata: counts[s].append(sum(1 for p in cal if s in mem[p]))
    else:
        pf = l.groupby('patient')['g'].agg(lambda s: tuple(s.dropna().unique()))
        pg = {p: (int(v[0]) if len(v) else 0) for p, v in pf.items()}
        pats = np.array(sorted(pg)); grp = np.array([pg[p] for p in pats])
        for _ in range(draws):
            t = set()
            for g in np.unique(grp):
                idx = np.where(grp == g)[0]; t.update(pats[r.choice(idx, int(round(0.4 * len(idx))), replace=False)])
            rm2 = np.array([p for p in pats if p not in t])
            cal = set(r.choice(rm2, int(round(0.6 * len(rm2))), replace=False))
            for s in strata: counts[s].append(sum(1 for p in cal if s in mem.get(p, ())))
    return pd.DataFrame([dict(cohort=cohort, stratum=f"{s[0]}{s[1] if s[0]=='M' else ''}",
                              ncal_mean=round(float(np.mean(v)), 1),
                              deg05=round(float(100 * np.mean(np.array(v) < 19)), 2)) for s, v in counts.items()])
A = pd.concat([audit_deg(hlesB, 'hiba'), audit_deg(plesB, 'pad')])
A.to_csv(f'{BASE}/results/akbenign_audit.csv', index=False)
print('AK-benign audit (paper reports: HIBA M1 3.9, HIBA M3 10.1, PAD M1 4.3, PAD B ~94 patients / 0):')
print(A.to_string(index=False))
ak = []
for d in range(100):
    r = np.random.default_rng(SEED * 1000 + d)
    cal = set(r.choice(hp, int(round(0.7 * len(hp))), replace=False))
    ak += [dict(cohort='hiba', draw=d, **x) for x in evaluate_draw2(hlesB, Sh, cal, set(hp) - cal, ALPHAS, r, MSETB)]
    calp = set(r.choice(rem, int(round(0.6 * len(rem))), replace=False))
    ak += [dict(cohort='pad', draw=d, **x) for x in evaluate_draw2(plesB, Sp, calp, set(rem) - calp, ALPHAS, r, MSETB)]
AK = pd.DataFrame(ak); AK.to_csv(f'{BASE}/results/akbenign_results.csv', index=False)
print(AK[AK.alpha == 0.05].groupby(['cohort', 'stratum'])[['coverage', 'referral', 'set_size']].mean().round(3).to_string())
print('\nDONE. 9 ablation files in Drive/conformal-triage/results/.')
